In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


# ============================================================
# Config
# ============================================================

RESULTS_DIR = Path("../results/scale_costs")
PLOTS_DIR = RESULTS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_MULTIPLIER = 1

# Only matters for raw dollar values.
# For ratios like profit / fruit_value, USD cancels out.
USD = 14500.0


# ============================================================
# Load raw JSON files
# ============================================================

def load_raw_results(results_dir: Path):
    """
    Load all JSON files in results_dir.

    Expected file structure:
        results/scale_n_ints/{n_id}.json

    Each file contains a list of results, one per n_ints value.
    """
    all_results = []

    json_paths = sorted(results_dir.glob("*.json"))

    if not json_paths:
        raise FileNotFoundError(f"No JSON files found in {results_dir}")

    for path in json_paths:
        with open(path, "r") as f:
            results = json.load(f)

        for result in results:
            result["_source_file"] = path.name
            all_results.append(result)

    return all_results


# ============================================================
# Process one raw result into one clean row
# ============================================================

def safe_get(dct, keys, default=np.nan):
    """
    Nested dictionary getter.
    """
    cur = dct
    for key in keys:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


def sum_values(dct):
    if not isinstance(dct, dict):
        return np.nan
    return np.sum([float(v) for v in dct.values()])


def sum_prob_values(dct):
    if not isinstance(dct, dict):
        return np.nan
    return np.sum([float(v) for v in dct.values()])


def build_scale_row(result, usd=USD):
    """
    Convert one raw simulation result into a compact analysis row.

    Economic metrics are normalized by total fruit value:
        total fruit value = sum farmer quantities * fruit price

    This controls for changing market size across n_ints.
    """
    sv = result["summary_vanilla"]

    max_sol = sv["max_int_welf_sol"]
    min_sol = sv["min_int_welf_sol"]

    farmer_quantities = result["farmer_quantities"]
    total_quantity = sum_values(farmer_quantities)

    # Prefer stored fruit_price from the instance.
    fruit_price = sv["instance"]["fruit_price"]
    fruit_value = total_quantity * fruit_price

    if fruit_value == 0 or pd.isna(fruit_value):
        fruit_value = np.nan

    matched_max = max_sol.get("matched_intermediaries", {})
    matched_min = min_sol.get("matched_intermediaries", {})

    n_ints = int(result["n_ints"])
    n_farmers = safe_get(sv, ["instance", "n_farmers"])
    n_intermediaries = safe_get(sv, ["instance", "n_intermediaries"])
    multiplier = float(result["multiplier"])

    row = {
        # -------------------------
        # Identifiers
        # -------------------------
        "source_file": result.get("_source_file"),
        "instance_id": result["instance_id"],
        "n_id": result["n_id"],
        "n_ints": n_ints,
        "multiplier": multiplier,

        # -------------------------
        # Instance scale
        # -------------------------
        "n_farmers": n_farmers,
        "n_intermediaries": n_intermediaries,
        "total_quantity": total_quantity,
        "fruit_price": fruit_price,
        "fruit_value": fruit_value / usd,

        # -------------------------
        # Raw economic values, in USD units if usd is set
        # -------------------------
        "profit_raw": max_sol["profit"] / usd,
        "farmer_welfare_raw": max_sol["farmer_welfare"] / usd,
        "intermediary_welfare_raw": max_sol["intermediary_welfare"] / usd,
        "total_welfare_raw": (
            max_sol["farmer_welfare"] + max_sol["intermediary_welfare"]
        ) / usd,
        "matching_cost_raw": max_sol["matching_cost"] / usd,

        # -------------------------
        # Main normalized economic metrics
        # share = metric / total fruit value
        # -------------------------
        "profit_share": max_sol["profit"] / fruit_value,
        "farmer_welfare_share": max_sol["farmer_welfare"] / fruit_value,
        "intermediary_welfare_share": max_sol["intermediary_welfare"] / fruit_value,
        "total_welfare_share": (
            max_sol["farmer_welfare"] + max_sol["intermediary_welfare"]
        ) / fruit_value,
        "matching_cost_share": max_sol["matching_cost"] / fruit_value,

        # -------------------------
        # Min-intermediary-welfare solution, if theoretically useful
        # -------------------------
        "min_int_profit_share": min_sol["profit"] / fruit_value,
        "min_int_farmer_welfare_share": min_sol["farmer_welfare"] / fruit_value,
        "min_int_intermediary_welfare_share": min_sol["intermediary_welfare"] / fruit_value,
        "min_int_total_welfare_share": (
            min_sol["farmer_welfare"] + min_sol["intermediary_welfare"]
        ) / fruit_value,
        "min_int_matching_cost_share": min_sol["matching_cost"] / fruit_value,

        # -------------------------
        # Optional forced/suboptimal quantities
        # -------------------------
        "forced_lower_bound_share": (
            sv["forced_lower_bound"] / fruit_value
            if sv.get("forced_lower_bound") is not None else np.nan
        ),
        "forced_upper_bound_share": (
            sv["forced_upper_bound"] / fruit_value
            if sv.get("forced_upper_bound") is not None else np.nan
        ),
        "forced_cost_share": (
            sv["forced_cost"] / fruit_value
            if sv.get("forced_cost") is not None else np.nan
        ),

        # -------------------------
        # Price coefficients
        # These are already coefficients, so do not divide by fruit value.
        # -------------------------
        "price_per_quantity": max_sol.get("price_per_quantity", np.nan),
        "price_per_mile_paved": max_sol.get("price_per_mile_paved", np.nan),
        "price_per_mile_dirt": max_sol.get("price_per_mile_dirt", np.nan),

        # -------------------------
        # Runtime / solver metrics
        # Do not divide these by fruit value.
        # -------------------------
        "time_vanilla": sv["total_time"],
        "oracle_calls": sv["total_oracle_calls"],

        # -------------------------
        # Matching saturation
        # -------------------------
        "n_matched_intermediaries": sum_prob_values(matched_max),
        "matched_share": sum_prob_values(matched_max) / n_ints,

        "min_int_n_matched_intermediaries": sum_prob_values(matched_min),
        "min_int_matched_share": sum_prob_values(matched_min) / n_ints,

        # -------------------------
        # Epsilon/cost summaries
        # -------------------------
        "avg_epsilon": np.mean(list(result["epsilon"].values())),
        "avg_cost": np.mean(list(result["cost"].values())),
    }

    # Computational metrics normalized by problem size
    if pd.notna(n_farmers) and n_farmers != 0:
        row["time_per_farmer"] = row["time_vanilla"] / n_farmers
        row["oracle_calls_per_farmer"] = row["oracle_calls"] / n_farmers
    else:
        row["time_per_farmer"] = np.nan
        row["oracle_calls_per_farmer"] = np.nan

    if pd.notna(n_farmers) and n_farmers != 0 and n_ints != 0:
        row["time_per_farmer_int_pair"] = row["time_vanilla"] / (n_farmers * n_ints)
        row["oracle_calls_per_farmer_int_pair"] = row["oracle_calls"] / (n_farmers * n_ints)
    else:
        row["time_per_farmer_int_pair"] = np.nan
        row["oracle_calls_per_farmer_int_pair"] = np.nan

    return row


def build_analysis_df(results_dir=RESULTS_DIR):
    raw_results = load_raw_results(results_dir)
    rows = [build_scale_row(result) for result in raw_results]
    df = pd.DataFrame(rows)

    # Clean dtypes
    df["n_id"] = df["n_id"].astype(str)
    df["n_ints"] = pd.to_numeric(df["n_ints"], errors="coerce")

    # Sort for readability
    df = df.sort_values(["n_id", "n_ints"]).reset_index(drop=True)

    return df


df = build_analysis_df(RESULTS_DIR)

print(df.shape)
print(df.head())

(1568, 43)
  source_file instance_id n_id  n_ints  multiplier  n_farmers  \
0      0.json      0_0.25    0      12        0.25         28   
1      0.json       0_0.5    0      12        0.50         28   
2      0.json      0_0.75    0      12        0.75         28   
3      0.json         0_1    0      12        1.00         28   
4      0.json      0_1.25    0      12        1.25         28   

   n_intermediaries  total_quantity  fruit_price  fruit_value  ...  \
0                12            47.3    2513000.0   8197.57931  ...   
1                12            47.3    2513000.0   8197.57931  ...   
2                12            47.3    2513000.0   8197.57931  ...   
3                12            47.3    2513000.0   8197.57931  ...   
4                12            47.3    2513000.0   8197.57931  ...   

   n_matched_intermediaries  matched_share  min_int_n_matched_intermediaries  \
0                       6.0            0.5                               6.0   
1                

In [2]:
def add_baseline_changes(df, metrics, base_multiplier=1.0):
    """
    For each metric, compute within-seed changes relative to base_n_ints.

    This respects the design:
        IID unit = n_id
        repeated sweep = n_ints within n_id
    """
    out = df.copy()

    out["multiplier"] = out["multiplier"].astype(float)
    out["multiplier"] = pd.to_numeric(out["multiplier"], errors="coerce")

    for metric in metrics:
        out[metric] = pd.to_numeric(out[metric], errors="coerce")

        base = (
            out.loc[out["multiplier"] == base_multiplier, ["n_id", metric]]
            .drop_duplicates(subset=["n_id"])
            .rename(columns={metric: f"{metric}_base"})
        )

        out = out.merge(base, on="n_id", how="left")

        base_col = f"{metric}_base"

        # Absolute paired change
        out[f"{metric}_diff_from_base"] = out[metric] - out[base_col]

        # Relative paired change, useful for runtime but less ideal for already-normalized shares
        out[f"{metric}_pct_change_from_base"] = np.where(
            out[base_col].abs() > 1e-12,
            100 * (out[metric] / out[base_col] - 1),
            np.nan,
        )

    return out

In [3]:
economic_share_metrics = [
    "profit_share",
    "farmer_welfare_share",
    "intermediary_welfare_share",
    "total_welfare_share",
    "matching_cost_share",
    "matched_share",
]

computational_metrics = [
    "time_vanilla",
    "oracle_calls",
    "time_per_farmer",
    "oracle_calls_per_farmer",
    "time_per_farmer_int_pair",
    "oracle_calls_per_farmer_int_pair",
]

all_metrics = economic_share_metrics + computational_metrics

df = add_baseline_changes(df, all_metrics, base_multiplier=1)

In [4]:
def paired_change_summary(df, change_metric, alpha=0.05):
    """
    Compute t-based confidence intervals across IID seeds
    for a within-seed change metric.
    """
    plot_df = df[["n_id", "multiplier", change_metric]].copy()

    plot_df["n_id"] = plot_df["n_id"].astype(str)
    plot_df["multiplier"] = pd.to_numeric(plot_df["multiplier"], errors="coerce")
    plot_df[change_metric] = pd.to_numeric(plot_df[change_metric], errors="coerce")

    plot_df = plot_df.dropna(subset=["n_id", "multiplier", change_metric])

    summary = (
        plot_df
        .groupby("multiplier")[change_metric]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
        .sort_values("multiplier")
    )

    summary["se"] = summary["std"] / np.sqrt(summary["n"])

    summary["tcrit"] = summary["n"].apply(
        lambda n: stats.t.ppf(1 - alpha / 2, df=n - 1) if n > 1 else np.nan
    )

    summary["ci_low"] = summary["mean"] - summary["tcrit"] * summary["se"]
    summary["ci_high"] = summary["mean"] + summary["tcrit"] * summary["se"]

    return summary

In [5]:
def plot_paired_change_ci(
    df,
    change_metric,
    ylabel=None,
    title=None,
    alpha=0.05,
    output_dir=PLOTS_DIR,
):
    summary = paired_change_summary(df, change_metric, alpha=alpha)

    x = summary["multiplier"].to_numpy(dtype=float)
    
    scale_y = 100 if change_metric.endswith("_diff_from_base") else 1
    mean = scale_y * summary["mean"].to_numpy(dtype=float)
    ci_low = scale_y * summary["ci_low"].to_numpy(dtype=float)
    ci_high = scale_y * summary["ci_high"].to_numpy(dtype=float)
    # mean = summary["mean"].to_numpy(dtype=float)
    # ci_low = summary["ci_low"].to_numpy(dtype=float)
    # ci_high = summary["ci_high"].to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(7, 4.5))

    ax.axhline(0, linewidth=1, linestyle="--", alpha=0.6)

    ax.plot(
        x,
        mean,
        marker="o",
        linewidth=3,
        label="Mean paired change",
    )

    ax.fill_between(
        x,
        ci_low,
        ci_high,
        alpha=0.25,
        label=f"{int((1 - alpha) * 100)}% CI for mean paired change",
    )

    ax.set_xlabel("Multiplier")
    ax.set_ylabel(ylabel or change_metric)
    ax.set_title(title or change_metric)
    ax.grid(alpha=0.3)
    ax.legend()

    fig.tight_layout()

    safe_name = change_metric.replace(".", "_").replace("/", "_")
    out_path = output_dir / f"scale_costs_{safe_name}_paired_ci.png"

    fig.savefig(out_path, dpi=300)
    plt.close(fig)

    print(f"Saved {out_path}")

In [6]:
pretty_labels = {
    "profit_share": "Profit / total fruit value",
    "farmer_welfare_share": "Farmer welfare / total fruit value",
    "intermediary_welfare_share": "Intermediary welfare / total fruit value",
    "total_welfare_share": "Total welfare / total fruit value",
    "matching_cost_share": "Matching cost / total fruit value",
    "matched_share": "Matched intermediaries / total intermediaries",
}

for metric in economic_share_metrics:
    change_metric = f"{metric}_diff_from_base"

    plot_paired_change_ci(
        df,
        change_metric,
        ylabel=f"Change from baseline\n(percentage points of total fruit value)",
        title=f"{pretty_labels.get(metric, metric)}: paired change from baseline",
    )

Saved ../results/scale_costs/plots/scale_costs_profit_share_diff_from_base_paired_ci.png
Saved ../results/scale_costs/plots/scale_costs_farmer_welfare_share_diff_from_base_paired_ci.png
Saved ../results/scale_costs/plots/scale_costs_intermediary_welfare_share_diff_from_base_paired_ci.png
Saved ../results/scale_costs/plots/scale_costs_total_welfare_share_diff_from_base_paired_ci.png
Saved ../results/scale_costs/plots/scale_costs_matching_cost_share_diff_from_base_paired_ci.png
Saved ../results/scale_costs/plots/scale_costs_matched_share_diff_from_base_paired_ci.png


In [20]:
summary = paired_change_summary(df, "profit_share_diff_from_base")
print(summary[["multiplier", "mean", "std", "n", "se", "ci_low", "ci_high"]])

   multiplier      mean       std    n        se    ci_low   ci_high
0        0.25 -0.015705  0.004655  196  0.000333 -0.016361 -0.015050
1        0.50 -0.010225  0.003021  196  0.000216 -0.010651 -0.009800
2        0.75 -0.005047  0.001494  196  0.000107 -0.005258 -0.004837
3        1.00  0.000000  0.000000  196  0.000000  0.000000  0.000000
4        1.25  0.004968  0.001470  196  0.000105  0.004761  0.005175
5        1.50  0.009888  0.002931  196  0.000209  0.009475  0.010301
6        1.75  0.014780  0.004392  196  0.000314  0.014161  0.015399
7        2.00  0.019649  0.005851  196  0.000418  0.018824  0.020473


In [21]:
summary = paired_change_summary(df, "matching_cost_share_diff_from_base")
print(summary[["multiplier", "mean", "std", "n", "se", "ci_low", "ci_high"]])

   multiplier      mean       std    n        se    ci_low   ci_high
0        0.25 -0.030967  0.005892  196  0.000421 -0.031797 -0.030137
1        0.50 -0.020649  0.003921  196  0.000280 -0.021201 -0.020097
2        0.75 -0.010331  0.001964  196  0.000140 -0.010608 -0.010054
3        1.00  0.000000  0.000000  196  0.000000  0.000000  0.000000
4        1.25  0.010312  0.001988  196  0.000142  0.010032  0.010592
5        1.50  0.020615  0.003939  196  0.000281  0.020060  0.021170
6        1.75  0.030941  0.005899  196  0.000421  0.030110  0.031772
7        2.00  0.041268  0.007860  196  0.000561  0.040161  0.042375


In [31]:
cols = [
    "multiplier",
    "intermediary_welfare_share",
    "min_int_intermediary_welfare_share",
    "farmer_welfare_share",
    "min_int_farmer_welfare_share",
]

display(df.groupby("multiplier")[cols[1:]].mean())

,intermediary_welfare_share,min_int_intermediary_welfare_share,farmer_welfare_share,min_int_farmer_welfare_share
multiplier,,,,
0.25,0.011866,0.001993,0.921176,0.931048
0.50,0.014614,0.001982,0.902629,0.915262
0.75,0.018146,0.002214,0.883601,0.899533
1.00,0.021757,0.002638,0.864612,0.883731
1.25,0.025241,0.003186,0.845849,0.867903
1.50,0.029288,0.003769,0.826578,0.852097
1.75,0.033190,0.004590,0.807458,0.836058
2.00,0.037222,0.005650,0.788230,0.819802
